# Wan 2.2 Image-to-Video — Official Repository (T4 Colab)

Built on the **official [`Wan-Video/Wan2.2`](https://github.com/Wan-Video/Wan2.2) GitHub repository** — native inference (`generate.py`), not the `diffusers` integration. `AutoPipelineForImage2Video` is never imported anywhere in this notebook.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`. This notebook restarts the runtime once (Step 2b) to apply dependency updates cleanly — when that happens, just run the remaining cells (or `Run all` again); it's expected, not an error.

**Model:** `Wan-AI/Wan2.2-TI2V-5B`, run via the official `ti2v-5B` task — the only Wan 2.2 checkpoint the repo positions for a single consumer-class GPU. The 14B `T2V-A14B` / `I2V-A14B` models are not used here: the official README states those need **at least 80GB VRAM** on a single GPU.

**On the output size — read this if you expected exactly 480x832:** the official repo hard-codes which sizes each task accepts (`wan/configs/__init__.py`'s `SUPPORTED_SIZES`). `480*832` is only valid for the 14B `i2v-A14B`/`t2v-A14B` tasks (the ones needing 80GB+ VRAM) — passing it to the T4-viable `ti2v-5B` task raises `AssertionError: Unsupport size 480*832 for task ti2v-5B`, confirmed by reading that file directly. `ti2v-5B`'s only 9:16 option is **`704*1280`**, used below instead so the notebook actually runs rather than crashing on an invalid combination.

**Honest T4 limitation:** the official README states TI2V-5B needs **at least 24GB VRAM** even with every memory-saving flag this notebook enables (`--offload_model True --convert_model_dtype --t5_cpu`). A free Colab T4 has 16GB. This notebook uses the smallest supported 9:16 size and the model's own tested default length (121 frames ≈ 5s @ 24fps) as the best-effort configuration most likely to fit, but it may still hit a CUDA out-of-memory error depending on what Colab allocates that session — `DURATION_SECONDS` is a plain variable in Step 5 if you need to shrink it further.

**`flash_attn` is intentionally skipped.** It's in the repo's `requirements.txt`, but its own attention module (`wan/modules/attention.py`) falls back to PyTorch's native `scaled_dot_product_attention` when `flash_attn` isn't installed — confirmed by reading that file directly. Compiling `flash_attn` in Colab is slow and failure-prone (a known issue the repo's own `INSTALL.md` calls out); skipping it trades a little speed for an install that actually finishes.

## 0. Confirm the GPU

In [ ]:
!nvidia-smi

## 1. Clone the official Wan 2.2 repository

In [ ]:
%cd /content
!git clone https://github.com/Wan-Video/Wan2.2.git
%cd /content/Wan2.2

## 2. Install only the required packages

Colab's preinstalled `torch` is already CUDA-matched to the T4 runtime, so it's left alone rather than reinstalled (reinstalling `torch` in Colab is a common way to accidentally break GPU support). `flash_attn` is filtered out for the reason above.

In [ ]:
!grep -viE '^(flash_attn|torch|torchvision|torchaudio)' requirements.txt > requirements_colab.txt
!pip install -q -r requirements_colab.txt
!pip install -q "huggingface_hub[cli]"

## 2b. Restart the runtime to apply dependency fixes

`pip install` above can upgrade packages (e.g. `numpy`, `transformers`) that Colab's base image already had a different version of pre-imported — a fresh process is the reliable way to make sure every cell after this one actually uses the new versions instead of silently keeping stale ones in memory.

**Running this cell intentionally crashes/restarts the runtime.** After it does: just continue running the notebook from Step 3 onward (or `Runtime -> Run all` again, from the top — the clone and installs above are quick to redo and safe to repeat).

In [ ]:
import os
print('Restarting the runtime to apply dependency updates — continue from Step 3 once it reconnects.')
os.kill(os.getpid(), 9)

## 3. Verify the environment after restart

In [ ]:
%cd /content/Wan2.2

import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU detected — set Runtime > Change runtime type > T4 GPU, then re-run.'
assert tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) >= (2, 4), \
    f'Wan 2.2 needs torch>=2.4.0, Colab has {torch.__version__}. Runtime > Disconnect and delete runtime, then re-run from the top.'

## 4. Download the T4-compatible Wan 2.2 model (TI2V-5B)

Several GB from Hugging Face — first run takes a few minutes.

In [ ]:
!huggingface-cli download Wan-AI/Wan2.2-TI2V-5B --local-dir ./Wan2.2-TI2V-5B

## 5. Upload your product image

In [ ]:
from google.colab import files

print('Choose one product photo to upload:')
uploaded = files.upload()  # saves into the current directory: /content/Wan2.2
image_filename = next(iter(uploaded))
print(f'Uploaded: {image_filename}')

## 6. Generate a 9:16 vertical video (official inference script)

`704*1280` is the `ti2v-5B` task's supported 9:16 size — see the size note at the top of this notebook. `DURATION_SECONDS` defaults to the model's own tested length; raise it if you have VRAM headroom, lower it first if you hit an out-of-memory error. Frame counts must be `4n+1`, handled automatically below.

In [ ]:
import subprocess

# --- Tunables ---------------------------------------------------------
SIZE = '704*1280'          # ti2v-5B's only 9:16 option (480*832 is not valid for this task)
FPS = 24                   # fixed by the ti2v-5B model config (sample_fps)
DURATION_SECONDS = 5       # ~121 frames — the model's own tested default; raise with caution on a T4
PROMPT = 'A realistic, premium commercial product video. Natural, smooth camera motion, cinematic lighting.'  # edit me
SAVE_FILE = 'output.mp4'
# ------------------------------------------------------------------------

FRAME_NUM = int(round((FPS * DURATION_SECONDS - 1) / 4)) * 4 + 1  # must be 4n+1
IMAGE_PATH = f'/content/Wan2.2/{image_filename}'

print(f'Requesting {FRAME_NUM} frames (~{FRAME_NUM / FPS:.1f}s @ {FPS}fps) at {SIZE}.')

result = subprocess.run(
    [
        'python', 'generate.py',
        '--task', 'ti2v-5B',
        '--size', SIZE,
        '--ckpt_dir', './Wan2.2-TI2V-5B',
        '--offload_model', 'True',
        '--convert_model_dtype',
        '--t5_cpu',
        '--image', IMAGE_PATH,
        '--prompt', PROMPT,
        '--frame_num', str(FRAME_NUM),
        '--save_file', SAVE_FILE,
    ],
    check=True,
)
print('Done:', SAVE_FILE)

## 7. Preview and download the MP4

In [ ]:
from IPython.display import Video, display
display(Video(SAVE_FILE, embed=True))

files.download("output.mp4")